# Gabriele's Notebook

In this notebook i test several models

In [ ]:
import pandas as pd
import numpy as np

# Plots
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import graph_objects as go

# Utils
from tqdm.notebook import tqdm
from tabulate import tabulate

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# Custom modules
import Models_regr as Models
import Dataset
import utils


# Set SEED
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Data loading

We first load the dataframe from the parquet file.

In [ ]:
df = pd.read_parquet('../Data/FI-2010-Cleaned/NoAuction/Zscore/WRT1V.parquet')
df.drop(columns=['Label_10', 'Label_20', 'Label_30', 'Label_50', 'Label_100', 'Day'], inplace=True)
df = df.head(100000)
target_cols = ['MP']

# Basic feature extraction
# df = utils.FeaturesExtractionLOB(df).extract()
df["MP"] = (df["P_Ask_1"] + df["P_Bid_1"]) / 2

df.head()

In [ ]:
# Split the dataframe into train and test
df_train, df_val, df_test = utils.DataTools.train_val_test_split(df)

# Standardize the data
scaler = StandardScaler()
scaler.set_output()
df_train[df_train.columns] = scaler.fit_transform(df_train)
df_val[df_val.columns] = scaler.transform(df_val)
df_test[df_test.columns] = scaler.transform(df_test)

# Split covariates and response variable(s)
X_train, y_train = utils.DataTools.split_x_y(df_train, target_cols)
X_val, y_val     = utils.DataTools.split_x_y(df_val, target_cols)
X_test, y_test   = utils.DataTools.split_x_y(df_test, target_cols)

# Build tensors 
X_train, y_train = utils.DataTools.numpy_to_tensor( X_train, y_train, device=device, dtype=torch.float32)
X_val, y_val     = utils.DataTools.numpy_to_tensor( X_val, y_val, device=device, dtype=torch.float32)
X_test, y_test   = utils.DataTools.numpy_to_tensor( X_test, y_test, device=device, dtype=torch.float32)

# Build TimeSeriesDataset
SEQ_LEN_X = 10
SEQ_LEN_Y = 1
OFFSET = 10
dataset_train = Dataset.TimeSeriesDataset(
    X         = X_train,
    y         = y_train,
    seq_len_x = SEQ_LEN_X,
    seq_len_y = SEQ_LEN_Y,
    offset    = OFFSET,
    channels  = True,
    task      = Dataset.TimeSeriesDataset.Task.REGRESSION
)
dataset_val = Dataset.TimeSeriesDataset(
    X         = X_val,
    y         = y_val,
    seq_len_x = SEQ_LEN_X,
    seq_len_y = SEQ_LEN_Y,
    offset    = OFFSET,
    channels  = True,
    task      = Dataset.TimeSeriesDataset.Task.REGRESSION
)
dataset_test = Dataset.TimeSeriesDataset(
    X         = X_test,
    y         = y_test,
    seq_len_x = SEQ_LEN_X,
    seq_len_y = SEQ_LEN_Y,
    offset    = OFFSET,
    channels  = True,
    task      = Dataset.TimeSeriesDataset.Task.REGRESSION
)

# Build TimeSeriesLoader
BATCH_SIZE = 32
dataloader_train = Dataset.TimeSeriesLoader(
    dataset = dataset_train,
    batch_size = BATCH_SIZE,
    shuffle = True
)
dataloader_val = Dataset.TimeSeriesLoader(
    dataset = dataset_val,
    batch_size = BATCH_SIZE,
    shuffle = False
)
dataloader_test = Dataset.TimeSeriesLoader(
    dataset = dataset_test,
    batch_size = 1024,
    shuffle = False
)

In [ ]:
print(f'Train dataset: {len(dataset_train)} samples')
print(f'Test dataset: {len(dataset_test)} samples')

## Models

### Aragosta

In [ ]:
dataloader_train.dataset.channels = True
dataloader_val.dataset.channels = True
dataloader_test.dataset.channels = True

In [ ]:
# Create the model
model_params = {
    'input_size': X_train.shape[1],
    'output_size': y_train.shape[1],
    'hidden_size': 64,
    'lstm_size': 64,
    'lstm_layers': 2,
    'conv_channels': 32,
    'inception_channels': 32
}
model = utils.ModelTools.build_model(Models.Aragosta, model_params).to(device)

# Optimizer parameters
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.MSELoss()

# Print the number of parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {n_params} parameters')

# Train the model
train_res = utils.train(
    train_loader = dataloader_train,
    val_loader = dataloader_val,
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    n_epochs = 3,
)

# Plot loss evolution
plt.figure(figsize=(10, 5))
plt.plot(train_res['losses_train'], label='Train loss')
plt.plot(train_res['losses_val'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
evaluation_res = utils.evaluate(
    model = model,
    loader = dataloader_test,
    criterion = criterion,
    device = device
)

true = np.concatenate(evaluation_res['true'], axis=0)
pred = np.concatenate(evaluation_res['pred'], axis=0)

print(f'sMAPE: {utils.Metrics.sMAPE(true,pred):.1f}%')
print(f'MSE: {torch.nn.MSELoss()(torch.tensor(true), torch.tensor(pred)).item():.4f}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(true)), y=true, mode='lines', name='True'))
fig.add_trace(go.Scatter(x=np.arange(len(pred)), y=pred, mode='lines', name='Predicted'))
fig.show()

### Basic LSTM

In [ ]:
dataloader_train.dataset.channels = False
dataloader_val.dataset.channels = False
dataloader_test.dataset.channels = False

In [ ]:
# Create the model
model_params = {
    'input_size': X_train.shape[1],
    'output_size': y_train.shape[1],
    'hidden_size': 64,
    'num_layers': 2,
    'dropout': 0.2
}
model = utils.ModelTools.build_model(Models.LSTM, model_params).to(device)

# Optimizer parameters
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.MSELoss()

# Print the number of parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {n_params} parameters')

# Train the model
train_res = utils.train(
    train_loader = dataloader_train,
    val_loader = dataloader_val,
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    n_epochs = 3,
)

# Plot loss evolution
plt.figure(figsize=(10, 5))
plt.plot(train_res['losses_train'], label='Train loss')
plt.plot(train_res['losses_val'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
evaluation_res = utils.evaluate(
    model = model,
    loader = dataloader_test,
    criterion = criterion,
    device = device
)

true = np.concatenate(evaluation_res['true'], axis=0)
pred = np.concatenate(evaluation_res['pred'], axis=0)

print(f'sMAPE: {utils.Metrics.sMAPE(true,pred):.1f}%')
print(f'MSE: {torch.nn.MSELoss()(torch.tensor(true), torch.tensor(pred)).item():.4f}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(true)), y=true, mode='lines', name='True'))
fig.add_trace(go.Scatter(x=np.arange(len(pred)), y=pred, mode='lines', name='Predicted'))
fig.show()

### Astice

<img src="../astice.png" width="200" >

In [ ]:
# Create the model
model_params = {
    'input_size': X_train.shape[1],
    'output_size': y_train.shape[1],
    'hidden_size': 64,
    'lstm_size': 64,
    'lstm_layers': 2,
    'smooth_conv_channels': 4
}
model = utils.ModelTools.build_model(Models.Astice, model_params).to(device)

# Optimizer parameters
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.MSELoss()

# Print the number of parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {n_params} parameters')

# Train the model
train_res = utils.train(
    train_loader = dataloader_train,
    val_loader = dataloader_val,
    model = model,
    optimizer = optimizer,
    criterion = criterion,
    device = device,
    n_epochs = 3,
)

# Plot loss evolution
plt.figure(figsize=(10, 5))
plt.plot(train_res['losses_train'], label='Train loss')
plt.plot(train_res['losses_val'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
evaluation_res = utils.evaluate(
    model = model,
    loader = dataloader_test,
    criterion = criterion,
    device = device
)

true = np.concatenate(evaluation_res['true'], axis=0)
pred = np.concatenate(evaluation_res['pred'], axis=0)

In [ ]:
print(f'sMAPE: {utils.Metrics.sMAPE(true,pred):.1f}%')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(true)), y=true, mode='lines', name='True'))
fig.add_trace(go.Scatter(x=np.arange(len(pred)), y=pred, mode='lines', name='Predicted'))
fig.show()